In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import KBinsDiscretizer, StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
# Load the data
data = pd.read_csv('Load Balancing Improved.csv')

# Preprocess the data
scaler = StandardScaler()
features = data[['task_size', 'cpu_demand', 'memory_demand', 'network_latency', 'io_operations', 'disk_usage', 'num_connections', 'priority_level']]
scaled_features = scaler.fit_transform(features)
target = data['target'].values

In [3]:
# Discretize the feature space
discretizer = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='uniform')
discretized_features = discretizer.fit_transform(scaled_features).astype(int)

e:\Python\lib\site-packages\sklearn\preprocessing\_discretization.py:239: FutureWarning: In version 1.5 onwards, subsample=200_000 will be used by default. Set subsample explicitly to silence this warning in the mean time. Set subsample=None to disable subsampling explicitly.
  warnings.warn(


In [4]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(discretized_features, target, test_size=0.2, random_state=42)

In [5]:
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(8542, 8) (2136, 8) (8542,) (2136,)


In [6]:

# Function to map states to unique integers
def state_to_index(state, bins):
    return np.ravel_multi_index(state, bins)

In [7]:
# Define the Q-learning agent
class QLearningAgent:
    def __init__(self, state_size, action_size, alpha=0.1, gamma=0.99, epsilon=1.0, epsilon_decay=0.995, epsilon_min=0.01):
        self.state_size = state_size
        self.action_size = action_size
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min
        self.q_table = np.zeros((state_size, action_size))

    def choose_action(self, state):
        if np.random.rand() <= self.epsilon:
            return np.random.choice(self.action_size)
        return np.argmax(self.q_table[state])

    def learn(self, state, action, reward, next_state):
        best_next_action = np.argmax(self.q_table[next_state])
        td_target = reward + self.gamma * self.q_table[next_state, best_next_action]
        td_error = td_target - self.q_table[state, action]
        self.q_table[state, action] += self.alpha * td_error

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay


In [8]:

# Define the environment
class Environment:
    def __init__(self, X, y):
        self.X = X
        self.y = y
        self.current_state = 0

    def reset(self):
        self.current_state = 0
        return tuple(self.X[self.current_state])

    def step(self, action):
        reward = -abs(self.y[self.current_state] - action)
        self.current_state += 1
        done = self.current_state >= len(self.y) - 1
        next_state = tuple(self.X[self.current_state]) if not done else tuple(self.X[0])
        return next_state, reward, done

# Initialize the agent and environment
state_size = 5 ** X_train.shape[1]  # Assuming 5 bins per feature
action_size = 10  # Assuming target values are discrete and range from 0 to 9
agent = QLearningAgent(state_size, action_size)
env = Environment(X_train, y_train)


In [9]:

# Train the agent
num_episodes = 500
for episode in range(num_episodes):
    state = state_to_index(env.reset(), [5] * X_train.shape[1])
    done = False
    while not done:
        action = agent.choose_action(state)
        next_state, reward, done = env.step(action)
        next_state_index = state_to_index(next_state, [5] * X_train.shape[1])
        agent.learn(state, action, reward, next_state_index)
        state = next_state_index

In [10]:
# Define the Q-learning agent
class QLearningAgent:
    def __init__(self, state_size, action_size, alpha=0.1, gamma=0.99, epsilon=1.0, epsilon_decay=0.995, epsilon_min=0.01):
        self.state_size = state_size
        self.action_size = action_size
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min
        self.q_table = np.zeros((state_size, action_size))

    def choose_action(self, state):
        if np.random.rand() <= self.epsilon:
            return np.random.choice(self.action_size)
        return np.argmax(self.q_table[state])

    def learn(self, state, action, reward, next_state):
        best_next_action = np.argmax(self.q_table[next_state])
        td_target = reward + self.gamma * self.q_table[next_state, best_next_action]
        td_error = td_target - self.q_table[state, action]
        self.q_table[state, action] += self.alpha * td_error

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

In [11]:

# Define the environment
class Environment:
    def __init__(self, X, y):
        self.X = X
        self.y = y
        self.current_state = 0

    def reset(self):
        self.current_state = 0
        return tuple(self.X[self.current_state])

    def step(self, action):
        reward = -abs(self.y[self.current_state] - action)
        self.current_state += 1
        done = self.current_state >= len(self.y) - 1
        next_state = tuple(self.X[self.current_state]) if not done else tuple(self.X[0])
        return next_state, reward, done


In [12]:
# Initialize the agent and environment
state_size = 5 ** X_train.shape[1]  # Assuming 5 bins per feature
action_size = 10  # Assuming target values are discrete and range from 0 to 9
agent = QLearningAgent(state_size, action_size)
env = Environment(X_train, y_train)

# Train the agent
num_episodes = 500
for episode in range(num_episodes):
    state = state_to_index(env.reset(), [5] * X_train.shape[1])
    done = False
    while not done:
        action = agent.choose_action(state)
        next_state, reward, done = env.step(action)
        next_state_index = state_to_index(next_state, [5] * X_train.shape[1])
        agent.learn(state, action, reward, next_state_index)
        state = next_state_index

# Evaluate the agent
def evaluate_agent(agent, env):
    state = state_to_index(env.reset(), [5] * X_train.shape[1])
    done = False
    total_reward = 0
    while not done:
        action = np.argmax(agent.q_table[state])
        next_state, reward, done = env.step(action)
        next_state_index = state_to_index(next_state, [5] * X_train.shape[1])
        total_reward += reward
        state = next_state_index
    return total_reward

# Evaluate on the test set
test_env = Environment(X_test, y_test)
total_reward = evaluate_agent(agent, test_env)
print(f'Total reward on test set: {total_reward}')

Total reward on test set: -512


In [13]:
import pickle
import joblib

# Save the trained Q-learning agent to a .pkl file using pickle
pickle_filename = 'q_learning_agent2.pkl'
with open(pickle_filename, 'wb') as file:
    pickle.dump(agent, file)

# Save the trained Q-learning agent to a .joblib file using joblib
joblib_filename = 'q_learning_agent2.joblib'
joblib.dump(agent, joblib_filename)

print(f"Agent saved to {pickle_filename} and {joblib_filename}")


Agent saved to q_learning_agent2.pkl and q_learning_agent2.joblib


In [14]:
# Save the trained Q-learning agent to a .pkl file using pickle
state_bins = {
        'task_size': [1, 25, 50, 75, 100],
        'cpu_demand': [0.1, 25, 50, 75, 100],
        'memory_demand': [1, 16, 32, 48, 64],
        'network_latency': [0.1, 50, 100, 150, 200],
        'io_operations': [1, 250, 500, 750, 1000],
        'disk_usage': [1, 25, 50, 75, 100],
        'num_connections': [1, 250, 500, 750, 1000],
        'priority_level': [1, 2, 3, 4, 5]
    }
pickle_filename = 'state_bins.pkl'
with open(pickle_filename, 'wb') as file:
    pickle.dump(state_bins, file)

In [ ]:
import json

# Evaluate the agent
def evaluate_agent(agent, env):
    state = state_to_index(env.reset(), [5] * X_train.shape[1])
    done = False
    total_reward = 0
    while not done:
        action = np.argmax(agent.q_table[state])
        next_state, reward, done = env.step(action)
        next_state_index = state_to_index(next_state, [5] * X_train.shape[1])
        total_reward += reward
        state = next_state_index
    return total_reward

# Evaluate on the test set
test_env = Environment(X_test, y_test)
total_reward = evaluate_agent(agent, test_env)

# Create a dictionary to store the metrics
metrics = {
    "total_reward": total_reward,
    "num_episodes": num_episodes,
    "state_size": state_size,
    "action_size": action_size,
    "alpha": agent.alpha,
    "gamma": agent.gamma,
    "epsilon": agent.epsilon,
    "epsilon_decay": agent.epsilon_decay,
    "epsilon_min": agent.epsilon_min
}

# Save the metrics to a JSON file
metrics_filename = 'model_metrics.json'
with open(metrics_filename, 'w') as file:
    json.dump(metrics, file, indent=4)

print(f"Metrics saved to {metrics_filename}")
